## This code will implement the Seq2seq model

In [92]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset,DataLoader
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence, pad_sequence
import pickle
import numpy as np
import torch, numpy as np
from torch.utils.data import Dataset
from torch.utils.data import random_split, DataLoader

## Pipeline comparaison

In [93]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [94]:
with open("/kaggle/input/new-seq2seq/Mapping.pkl", "rb") as f:
    mapping = pickle.load(f)

In [95]:
class MMapSeq2Seq(Dataset):
    def __init__(self, prefix=""):
        self.ctx = np.load(prefix+"context_np.npy","r")
        self.x   = np.load(prefix+"X_np.npy","r")
        self.y   = np.load(prefix+"Y_np.npy","r")

        self.ctx_off = np.load(prefix+"context_offset.npy")
        self.ctx_len = np.load(prefix+"context_length.npy")
        self.x_off   = np.load(prefix+"X_offset.npy")
        self.x_len   = np.load(prefix+"X_length.npy")
        self.y_off   = np.load(prefix+"Y_offset.npy")
        self.y_len   = np.load(prefix+"Y_length.npy")

        assert len(self.ctx_len) == len(self.x_len) == len(self.y_len)

    def __len__(self):
        return len(self.ctx_len)

    def __getitem__(self, i):
        # vues memmap -> torch (zéro copie CPU)
        s, L = int(self.ctx_off[i]), int(self.ctx_len[i])
        ctx = torch.from_numpy(self.ctx[s:s+L].astype(np.int32))

        s, L = int(self.x_off[i]), int(self.x_len[i])
        x = torch.from_numpy(self.x[s:s+L].astype(np.int32))

        s, L = int(self.y_off[i]), int(self.y_len[i])
        y = torch.from_numpy(self.y[s:s+L].astype(np.int32))

        return ctx, x, y

In [96]:
def collate(batch):
    ctxs, xs, ys = zip(*batch)  # tuples de 1D tensors CPU
    ctxs = [t.to(torch.long, copy=False) for t in ctxs]
    xs   = [t.to(torch.long, copy=False) for t in xs]
    ys   = [t.to(torch.long, copy=False) for t in ys]

    ctx_pad = torch.nn.utils.rnn.pad_sequence(ctxs, batch_first=True, padding_value=PAD_ID)
    x_pad   = torch.nn.utils.rnn.pad_sequence(xs,   batch_first=True, padding_value=PAD_ID)
    y_pad   = torch.nn.utils.rnn.pad_sequence(ys,   batch_first=True, padding_value=PAD_ID)

    ctx_len = torch.tensor([t.numel() for t in ctxs], dtype=torch.int64)  # CPU Long (pack_padded)
    x_len   = torch.tensor([t.numel() for t in xs],   dtype=torch.int64)

    return (ctx_pad, x_pad, y_pad), (ctx_len, x_len)

It seems like Kaggle really dislike my first method in term of computation : time divided by x1000

I think it's because the format was dataset so pretty heavy compared to np and also I'm using "r" that allow to only load what I need

### Now, pipeline prep

In [97]:
PAD_ID = 0
ds = MMapSeq2Seq(prefix="/kaggle/input/new-seq2seq/")  # mets des préfixes train/val si tu split offline

In [98]:
train_size = int(0.8 * len(ds))
test_size   = len(ds) - train_size
train_ds, test_ds = random_split(ds, [train_size, test_size])

batch_size = 128

In [99]:
if device == "cpu" :
    train_ld = DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=True, collate_fn=collate) #Shuffle False because we need the RNN to use previous sequences data to predict next one
    test_ld = DataLoader(test_ds, batch_size=batch_size, shuffle=False, drop_last=True, collate_fn=collate)
else : 
    train_ld = DataLoader(train_ds, batch_size=batch_size, pin_memory=True, pin_memory_device=device, 
                        num_workers=4, shuffle=True, drop_last=True, persistent_workers=False, collate_fn=collate) #Shuffle False because we need the RNN to use previous sequences data to predict next one
    test_ld = DataLoader(test_ds, batch_size=batch_size, pin_memory=True, pin_memory_device=device, 
                        num_workers=2, shuffle=True, drop_last=True, persistent_workers=False, collate_fn=collate)

List of operations :

- First the entire batch pass through the Encodeur, we extract the h,c from this

- Then we iterate over all the length of the sequence, with the previous input and we use at first the h,c from the Encodeur

- Then we calculate the loss

## Create Seq2seq

In [100]:
class Encodeur(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size, num_layers=1):
        super(Encodeur, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        self.embed = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(input_size = embedding_dim, 
                            hidden_size = hidden_size, 
                            num_layers = num_layers, 
                            dropout = 0.25,
                            batch_first=True)

    def forward(self, x, length_batch):
        #Encodeur
        emb_x = self.embed(x)
        #Pack before LSTM to avoid unecessary compute
        packed_x = pack_padded_sequence(emb_x, length_batch, batch_first=True, enforce_sorted=False)                      
        packed_out, (h,c) = self.lstm(packed_x)             
        #Unpack after
        _, _ = pad_packed_sequence(packed_out, batch_first=True) #First right now only h,c
        return (h, c)

In [101]:
class Decodeur(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size, masked_mapping, num_layers=1):
        super(Decodeur, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        self.embed = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(input_size = embedding_dim, 
                            hidden_size = hidden_size, 
                            num_layers = num_layers, 
                            dropout = 0.25,
                            batch_first=True)
        
        self.final = nn.Linear(hidden_size,vocab_size)
        self.mask = masked_mapping

#    def forward(self, x, h, c, length_batch):
    def forward(self, x, h, c, length_batch, targets=None, teacher_forcing_ratio=1.0):
        batch_size, max_len = targets.size()
        #batch_size, max_len = Y.size() 
        device = x.device

        # On stockera les logits ici
        all_logits = []

        # Premier input : <SOS> pour tout le batch
        input_t = x[:, 0]

        for t in range(0, max_len):
            emb = self.embed(input_t).unsqueeze(1)  # (batch, 1, emb_dim)
            out, (h, c) = self.lstm(emb, (h, c))    # (batch, 1, hidden)
            logit = self.final(out.squeeze(1))      # (batch, vocab_size)
            masked_logit = logit.masked_fill(self.mask, float("-inf"))
            all_logits.append(masked_logit.unsqueeze(1))

            # Decide teacher forcing ou pas
            if torch.rand(1).item() < teacher_forcing_ratio:
                input_t = targets[:, t]   # vérité
            else:
                input_t = masked_logit.argmax(dim=-1)  # prédiction

        all_logits = torch.cat(all_logits, dim=1)  # (batch, max_len-1, vocab_size)
        return all_logits, (h, c)
#        emb_x = self.embed(x)
        #Pack before LSTM to avoid unecessary compute
#        packed_x = pack_padded_sequence(emb_x, length_batch, batch_first=True, enforce_sorted=False)                      
#        x_lstm, (h,c) = self.lstm(packed_x, (h,c))             
        #Unpack 
#        unpacked_out,_ = pad_packed_sequence(x_lstm, batch_first=True) 
#        logits = self.final(unpacked_out)
#        masked_logits = logits.masked_fill(self.mask, float("-inf"))
  
#        return masked_logits, (h, c)

In [111]:
vocab_size = len(mapping)
embedding_size = 64
hidden_size = 512
num_epoch = 100

#Some of the mapping are only for the encodeur so the decodeur can't produce them, we need to mask them from the loss
mapping_inverse = {i: ch for ch, i in mapping.items()}
masked_mapping = list(mapping_inverse.keys())[111:-1]

mask = torch.zeros(vocab_size, dtype=torch.bool, device=device)
mask[masked_mapping] = True

nb_step_test = len(test_ld)
nb_step_train = len(train_ld)

enco = Encodeur(vocab_size, embedding_size, hidden_size, num_layers=2).to(device)
deco = Decodeur(vocab_size, embedding_size, hidden_size, mask,num_layers=2).to(device)

loss_fn = nn.CrossEntropyLoss(ignore_index=0) #Just ignore the padding

params = list(enco.parameters()) + list(deco.parameters())
opti = torch.optim.AdamW(params, lr=0.005, betas=(0.9,0.999), weight_decay=1e-3)

sched_warm = torch.optim.lr_scheduler.LinearLR(opti, start_factor=0.2, end_factor=1.0, total_iters=nb_step_train*2)
sched_post = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(opti, T_0=nb_step_train*10, T_mult=2, eta_min=0.001) #1 epoch => 2 => 4 => 8
#scheduler = torch.optim.lr_scheduler.SequentialLR(
#    opti,
#    schedulers=[sched_warm, sched_post],
#    milestones=[nb_step_train*2],
#)

## Creation boucle entraînement Decodeur

In [112]:
def token_accuracy(logits, Y, pad_id=0):
    # logits: [B, T, V], Y: [B, T]
    preds = logits.argmax(dim=-1)                 # [B, T]
    mask  = (Y != pad_id)                         # [B, T]
    correct = (preds == Y) & mask
    return correct.sum().float() / mask.sum().float()

def topk_token_accuracy(logits, Y, k=5, pad_id=0):
    """
    logits : (B, T, V)
    Y      : (B, T)
    """
    # tronquer Y si jamais logits et Y diffèrent un peu
    Y = Y[:, :logits.size(1)]

    # top-k indices
    topk = logits.topk(k, dim=-1).indices  # (B, T, k)

    # mask pour ignorer les PAD
    mask = (Y != pad_id)

    # comparaison : Y est-il dans top-k ?
    correct = (topk == Y.unsqueeze(-1)).any(dim=-1) & mask  # (B, T)

    return correct.sum().float() / mask.sum().float()


In [113]:
l_tot = []
#Early stopping
#early_stopping_count = 0
#patience = 5
best_val = float("inf")

scaler = torch.amp.GradScaler()

for epoch in range(num_epoch) :
    
    enco.train()
    deco.train()

    #Create loss per epoch
    l_train = 0.0
    l_test = 0.0

    teacher_forcing_ratio = max(0.5, 1.0 - (max(0,epoch-25)) * 0.01)
    if epoch%10==0:
        print(teacher_forcing_ratio)
    
    for (context,X, Y), (length_cont,length_text) in iter(train_ld) :
        X = X.to(device, non_blocking = True)
        Y = Y.to(device, non_blocking = True)
        context = context.to(device, non_blocking = True)
#        length_text = (Y != 0).sum(dim=1).cpu()
        opti.zero_grad(set_to_none=True)

        #Encodeur part, sees the whole context of the batch and output the hidden and cell state
        h_enco,c_enco = enco(context,length_cont)
        
        h_dec = h_enco
        c_dec = c_enco
        
        with torch.amp.autocast(device_type="cuda"):
            logits, (h_dec, c_dec) = deco(
            X, h_dec, c_dec, length_text, 
            targets=Y, teacher_forcing_ratio=teacher_forcing_ratio   # <--- par ex.
            )
            loss = loss_fn(logits.reshape(-1, vocab_size), Y.reshape(-1))
            #logits, (h_dec, c_dec) = deco(X,h_dec,c_dec,length_text)
            #loss = loss_fn(logits.reshape(-1,vocab_size),Y.reshape(-1))

        scaler.scale(loss).backward()
        #Gradient clipping to avoid exploding gradient
        scaler.unscale_(opti)
        torch.nn.utils.clip_grad_norm_(list(enco.parameters()) + list(deco.parameters()), 0.25)

        #Scaler
        scaler.step(opti); scaler.update()
        l_train += loss.item()

        #Scheduler part
        #Warm start
        
        if sched_post.T_cur == 0 and epoch > 2:  #After warm restart decrease the max learning rate
            sched_post.base_lrs[0] = sched_post.base_lrs[0] * 0.6
            sched_post.eta_min = sched_post.eta_min * 1
            print(f"Decrease {sched_post.base_lrs[0]}, {sched_post.eta_min}")

        step_scheduler = sched_warm if epoch < 2 else sched_post
        step_scheduler.step()

    #Test data part
    
    enco.eval()
    deco.eval()

    acc_sum, tok_sum = 0.0, 0
    topk_acc_sum = 0.0    
    
    with torch.inference_mode():
        with torch.amp.autocast("cuda"):
            for (context, X, Y), (length_cont,length_text) in iter(test_ld) :
                X = X.to(device, non_blocking = True)
                Y = Y.to(device, non_blocking = True)
                context = context.to(device, non_blocking = True)

                #Encodeur part, sees the whole context of the batch and output the hidden and cell state
                h_enco,c_enco = enco(context,length_cont)
        
                h_dec = h_enco
                c_dec = c_enco
                
                logits, (h_dec, c_dec) = deco(X, h_dec, c_dec, length_text, targets=Y, teacher_forcing_ratio=1)
                loss = loss_fn(logits.reshape(-1, vocab_size), Y.reshape(-1))

                #logits, (h_dec, c_dec) = deco(X,h_dec,c_dec,length_text)
                #loss = loss_fn(logits.reshape(-1,vocab_size),Y.reshape(-1))
                l_test += loss.item()
                acc = token_accuracy(logits, Y).item()
                
                # top-1 accuracy
                acc_sum += ((logits.argmax(-1) == Y) & (Y != 0)).sum().item()
                tok_sum += (Y != 0).sum().item()

                # top-k accuracy (par ex. k=5)
                topk_acc_sum += topk_token_accuracy(logits, Y, k=5).item()

        epoch_token_acc = acc_sum / tok_sum
        epoch_topk_acc  = topk_acc_sum / nb_step_test   # moyenne sur les batches

        print(epoch, np.exp(l_test/nb_step_test), epoch_token_acc, epoch_topk_acc, "\n")
     
        #Record the loss of the epoch
        l_tot.append(l_test); 

        if l_test < best_val :
            best_val = l_test
#            early_stopping_count = 0
            torch.save({
                "epoch": epoch,
                "encoder_state_dict": enco.state_dict(),
                "decoder_state_dict": deco.state_dict(),
                "optimizer_state_dict": opti.state_dict(),
                "scheduler_state_dict": sched_post.state_dict(),
                "val_loss": l_test,
            }, "model1")
        
#        elif l_test >= best_val :
#            early_stopping_count += 1

#        if early_stopping_count == patience :
#            print("Early Stopping")
#            break 

print(f"Liste of offset used : {list_offset}")

1.0
0 18.44039418695396 0.22618887694787748 0.5532009431294033 

1 10.29944878524181 0.32805531275913846 0.7099267670086452 

2 7.711357654074633 0.4059317291550084 0.7620439529418945 

3 6.518286143091919 0.44709668015831194 0.7892401814460754 

4 5.799844440847546 0.48124496734365213 0.8096736328942435 

5 5.382369552952844 0.5010833631165118 0.8212971602167402 

6 5.105064485929395 0.516923111556341 0.8296393496649606 

7 4.875911398365327 0.5291432662833105 0.8350109117371696 

8 4.741496093926367 0.5380859593719904 0.8395298038210187 

9 4.632390620960681 0.5444132509825941 0.8440251350402832 

1.0
10 4.582882524936152 0.5471221317208547 0.8447410293987819 

11 4.520060929498837 0.5514243350107836 0.8468339102608817 

Decrease 0.003, 0.001
12 4.497407374231002 0.5522548438007185 0.8460800221988133 

13 4.387671137782167 0.5597322809928473 0.851353108882904 

14 4.3488173993581825 0.5623433864327904 0.851299021925245 

15 4.2888224793835175 0.5690965034965035 0.8528551118714469 

1

KeyboardInterrupt: 